# TIB P4.1 — controlled development trials

Prepared notebook; **not executed**. Choose T4. Upload matching ZIP to private `MyDrive/TIB_P41_DEVELOPMENT_261019/`. Default7 profiles produce at most21 neural outputs on the same3 sealed pilot sources. BART1 + LED3 + Qwen3; no automatic repeats/search. Reserve512 is deferred and excluded. No references, selection300, final523 or new cohort records. Cap350 is a maximum. Flags are diagnostic; formal profile/acceptance freeze remains pending. Qwen larger reserves also change windows: compare complete pipelines, not an isolated reserve effect.

Baseline ZIP and checkpoints stay untouched. Each trial has independent IDs and checkpoints. New code/seed policy means archive-vs-new outputs are not a matched RNG replication. In this notebook seed1729 maps source/stage/layer/node consistently across trials. CPU scorer readiness already exists; formal-runtime scorer, fresh-session resume, fullpilot40, quotas and selection remain separate later stages. Session guard11.5h includes setup. Do not reset its timer to bypass the deadline.

Final export includes runner code, profile/registry/rules, three source-only records with attribution, receipts/logs/node outputs. Keep the evidence private.

In [ ]:
import pathlib,time,sys,subprocess,json,hashlib,zipfile,os,shutil
if 'P41_SESSION_STARTED' not in globals(): P41_SESSION_STARTED=time.monotonic()
P41_GUARD_SECONDS=11.5*3600
from google.colab import drive
drive.mount('/content/drive')
ROOT=pathlib.Path('/content/drive/MyDrive/TIB_P41_DEVELOPMENT_261019')
ZIP_PATH=ROOT/'tib-p41-development-261019.zip'
STORE=ROOT/'checkpoints'
assert ZIP_PATH.is_file(),f'Upload matching ZIP first: {ZIP_PATH}'
STORE.mkdir(parents=True,exist_ok=True)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','transformers==4.57.6','accelerate>=1,<2','safetensors','psutil','scikit-learn'],check=True)
import torch
assert torch.cuda.is_available(),'Choose a GPU runtime'
print('Actual GPU:',torch.cuda.get_device_name(0))
assert torch.cuda.get_device_name(0) in ('Tesla T4','NVIDIA T4','NVIDIA Tesla T4'),'Run the controlled trials on T4 for T4 evidence' 

In [ ]:
assert hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()=='51593e37d868ba05d5fb1c2dfb2ef8eaa95467ca8b6ff1ddeac56b7dd3efefa8','Wrong ZIP version'
BUNDLE=pathlib.Path('/content/tib_p41_development_261019')
BUNDLE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    for name in archive.namelist():
        assert (BUNDLE/name).resolve().is_relative_to(BUNDLE.resolve()),'Unsafe ZIP member'
    archive.extractall(BUNDLE)
manifest=json.loads((BUNDLE/'bundle_manifest.json').read_text())
for name,expected in manifest['files'].items():
    assert hashlib.sha256((BUNDLE/name).read_bytes()).hexdigest()==expected,name
registry=json.loads((BUNDLE/'development_trials.json').read_text())
os.chdir(BUNDLE)
sys.path.insert(0,str(BUNDLE))
# Copy the immutable input code/registry/source-only bundle into exported evidence.
for name in [*manifest['files'],'bundle_manifest.json']:
    target=ROOT/'run_bundle'/name
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists(): assert target.read_bytes()==(BUNDLE/name).read_bytes(),'Do not mix bundle versions'
    else: shutil.copyfile(BUNDLE/name,target)
print('Verified staged IDs:',manifest['staged_pilot_ids'])
print('Default trials:',[t['trial_id'] for t in registry['trials'] if t['enabled_by_default']])

In [ ]:
# Separate process per trial: no model/scorer GPU concurrency or hidden retries.
for trial in registry['trials']:
    if not trial['enabled_by_default']: continue
    remaining=P41_GUARD_SECONDS-(time.monotonic()-P41_SESSION_STARTED)-60
    if remaining<=60:
        print('Session guard reached; retain partial evidence and resume in a new session');break
    command=[sys.executable,'-u','-m','experiments.scripts.tib_benchmark_colab_capacity',
             '--bundle',str(BUNDLE),'--store',str(STORE),'--candidate',trial['candidate_id'],
             '--trial-id',trial['trial_id'],'--wall-hours',str(min(11.0,remaining/3600))]
    log=STORE/(trial['trial_id']+'-session-'+str(time.time_ns())+'.log')
    try:
        with log.open('w') as handle:
            result=subprocess.run(command,cwd=BUNDLE,stdout=handle,stderr=subprocess.STDOUT,timeout=remaining)
        print(trial['trial_id'],'exit',result.returncode,'log:',log)
    except subprocess.TimeoutExpired:
        print('Deadline interrupted trial; keep attempts and checkpoints:',log);break
# Deferred qwen-r512 requires separate resource review and verified qwen-r384 T4
# receipt with >=1GiB measured reserved headroom. Not enabled by this notebook.

In [ ]:
from experiments.scripts.tib_benchmark_checkpoint import restore
import pandas as pd
summary=[]
for path in sorted(STORE.rglob('capacity-*.json')):
    r=restore(path)
    events=[e for record in r['records'] for e in record.get('attempts',[])]
    native=[e['backend_receipt'] for e in events if e.get('backend_receipt')]
    summary.append({'trial_id':r.get('trial_id'),'GPU':r['device_name'],'status':r['status'],
       'T4_three_source_capacity':r['T4_feasibility_verified'],
       'VRAM_reserved_GiB':r.get('cuda_peak_reserved_bytes',0)/1024**3,
       'new_node_seconds':sum(e.get('new_attempt_wall_seconds',0) for e in events),
       'reused_nodes':sum(e['status']=='reused' for e in events),
       'exhausted_without_EOS':sum(n.get('token_budget_exhausted') is True for n in native),
       'repetition_flags':sum(n.get('repetition_flag') is True for n in native),
       'sentence_ending_flags':sum(n.get('sentence_completion_flag') is True for n in native),
       'wall_seconds':r.get('total_wall_seconds'),'receipt':str(path)})
display(pd.DataFrame(summary))
print('Diagnostic trial evidence only. No automatic quality winner/admission. Original time on reused nodes is separate from new-call cost.')

In [ ]:
# Export code + source-only inputs + all attempts/logs/receipts/outputs.
from datetime import datetime,timezone
from google.colab import files
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
EXPORT=pathlib.Path('/content')/('tib-p41-evidence-'+stamp+'.zip')
paths=sorted(p for folder in (ROOT/'run_bundle',STORE) for p in folder.rglob('*') if p.is_file())
inventory=[]
try:
    with zipfile.ZipFile(EXPORT,'x',zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for path in paths:
            assert path.resolve().is_relative_to(ROOT.resolve())
            before=path.stat()
            data=path.read_bytes()
            after=path.stat()
            assert (before.st_size,before.st_mtime_ns)==(after.st_size,after.st_mtime_ns),'Wait for active writers'
            name=path.relative_to(ROOT).as_posix()
            archive.writestr(name,data)
            inventory.append({'path':name,'bytes':len(data),'sha256':hashlib.sha256(data).hexdigest()})
        assert paths==sorted(p for folder in (ROOT/'run_bundle',STORE) for p in folder.rglob('*') if p.is_file()),'Active writer added files'
        archive.writestr('evidence_manifest.json',json.dumps({'created_at_utc':stamp,
            'scope':'P4.1 controlled three-source development; no formal benchmark',
            'file_count':len(inventory),'total_uncompressed_bytes':sum(x['bytes'] for x in inventory),
            'files':inventory},indent=2))
    with zipfile.ZipFile(EXPORT) as archive: assert archive.testzip() is None
except Exception:
    EXPORT.unlink(missing_ok=True)
    raise
print('Evidence ZIP:',EXPORT,'files:',len(inventory))
files.download(str(EXPORT))